# D4 Transportation Workflow — GeoPrompt

Network routing, service area coverage, and mobility scenario analysis.


In [3]:
from __future__ import annotations

import json, os
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import geoprompt as gp

_ROOT = Path.cwd()
if (_ROOT / "examples" / "notebooks").exists():
    OUTPUT_DIR = _ROOT / "examples" / "notebooks" / "geoprompt" / "outputs"
else:
    OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALLOW_LIVE_API = os.getenv("GEOPROMPT_ALLOW_LIVE_API", "0") == "1"


def fetch_json(url, fallback):
    if not ALLOW_LIVE_API:
        return fallback
    try:
        req = Request(url, headers={"User-Agent": "geoprompt-notebook/2.0"})
        with urlopen(req, timeout=6) as r:
            return json.loads(r.read().decode("utf-8"))
    except (URLError, TimeoutError, ValueError):
        return fallback


def fetch_first_json(urls, validator, fallback):
    for url in urls:
        payload = fetch_json(url, None)
        if payload is not None and validator(payload):
            return payload, url, True
    return fallback, "fallback", False


def make_point(x: float, y: float):
    return {"type": "Point", "coordinates": [float(x), float(y)]}


def frame_from_xy(data: dict[str, list], xs: list[float], ys: list[float], crs: str = "EPSG:4326"):
    columns = list(data.keys())
    rows = []
    for i, (x, y) in enumerate(zip(xs, ys)):
        row = {col: data[col][i] for col in columns}
        row["geometry"] = make_point(x, y)
        rows.append(row)
    return gp.GeoPromptFrame.from_records(rows, crs=crs)


def frame_preview(frame: gp.GeoPromptFrame, columns: list[str], limit: int | None = None):
    rows = frame.to_records()
    if limit is not None:
        rows = rows[:limit]
    return pd.DataFrame([{col: row.get(col) for col in columns} for row in rows], columns=columns)


print(f"geoprompt {gp.__version__} ready")

geoprompt 0.2.0 ready


## Section A: Pull Data Sources


In [2]:
transport = {"features": [{"id": "fallback-transport"}]}
weather = {"properties": {"forecast": "fallback"}}
forecast = {"hourly": {"temperature_2m": [0.0]}}

transport, tr_src, tr_live = fetch_first_json(
    [
        "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson",
        "https://api.github.com/repos/osm-search/Nominatim",
    ],
    lambda d: isinstance(d, dict) and bool(d.get("features") or d.get("id")),
    transport,
)
weather, wx_src, wx_live = fetch_first_json(
    [
        "https://api.weather.gov/points/40.75,-111.90",
        "https://api.weather.gov/points/34.05,-118.24",
    ],
    lambda d: isinstance(d, dict) and bool(d.get("properties", {}).get("forecast")),
    weather,
)
forecast, fc_src, fc_live = fetch_first_json(
    [
        "https://api.open-meteo.com/v1/forecast?latitude=40.75&longitude=-111.90&hourly=temperature_2m&forecast_days=1",
        "https://api.open-meteo.com/v1/forecast?latitude=34.05&longitude=-118.24&hourly=temperature_2m&forecast_days=1",
    ],
    lambda d: isinstance(d, dict) and len(d.get("hourly", {}).get("temperature_2m", [])) > 0,
    forecast,
)

transport_count = len(transport.get("features", [])) if isinstance(transport, dict) else 0
if transport_count == 0 and isinstance(transport, dict) and transport.get("id"):
    transport_count = 1
print(f"Transport records: {transport_count} | live={tr_live} | source={tr_src}")
print(f"NOAA forecast exists: {bool(weather.get('properties', {}).get('forecast'))} | live={wx_live} | source={wx_src}")
print(f"Open-Meteo hourly points: {len(forecast.get('hourly', {}).get('temperature_2m', []))} | live={fc_live} | source={fc_src}")


Transport records: 1 | live=False | source=fallback
NOAA forecast exists: True | live=False | source=fallback
Open-Meteo hourly points: 1 | live=False | source=fallback


## Section B: Spatial Analysis


In [4]:
nodes_data = {
    "node":       ["O",     "A",     "B",     "C",     "D",     "E",     "F"],
    "cost":       [0.0,     5.0,     9.0,     12.0,    14.0,    11.0,    3.0],
    "throughput": [1500,    1200,    1100,    900,     800,     950,     1300],
    "node_type":  ["origin", "inter", "inter", "dest", "inter", "inter", "inter"],
}

lons = [-111.90, -111.87, -111.84, -111.78, -111.84, -111.81, -111.86]
lats = [  40.75,   40.75,   40.75,   40.75,   40.72,   40.72,   40.78]

gdf = frame_from_xy(nodes_data, lons, lats, crs="EPSG:4326")

print("Transport nodes:")
print(frame_preview(gdf, ["node", "cost", "throughput"]).to_string(index=False))


# Incident data
incidents_gdf = frame_from_xy(
    {"inc_id": ["I1", "I2", "I3"], "severity": [3, 1, 2]},
    [-111.88, -111.83, -111.79],
    [40.76, 40.73, 40.75],
    crs="EPSG:4326",
)


# 1. Buffer: service zones around nodes (projected)
gdf_proj = gdf.to_crs("EPSG:3857")
service_buffers = gdf.select_columns(["node", "cost"]).to_crs("EPSG:3857").buffer(3000).to_crs("EPSG:4326")
print(f"\nService buffers (3km): {len(service_buffers)}")


# 2. Nearest join: assign incidents to nearest node (km)
incidents_m = incidents_gdf.to_crs("EPSG:3857")
nodes_m = gdf.to_crs("EPSG:3857")
nearest_m = incidents_m.nearest_join(nodes_m, how="left", rsuffix="node")
nearest = nearest_m.to_crs("EPSG:4326")
nearest["dist_km"] = [None if d is None else d / 1000.0 for d in nearest_m["distance_node"]]

print("\nIncidents assigned to nearest node (km):")
print(frame_preview(nearest, ["inc_id", "severity", "node", "dist_km"]).to_string(index=False))


# 3. Spatial join: incidents within service buffers
in_buf = incidents_gdf.spatial_join(service_buffers.select_columns(["node", "cost"]), how="left", predicate="within", rsuffix="svc")
print("Incidents within 3km service zones:")
node_col = "node" if "node" in in_buf.columns else [c for c in in_buf.columns if c.startswith("node")][0]
print(frame_preview(in_buf, ["inc_id", node_col]).dropna().to_string(index=False))


# 4. Dissolve by node type
dissolved = (
    pd.DataFrame(gdf.to_records())
    .groupby("node_type", as_index=True)
    .agg({"throughput": "sum", "cost": "mean"})
)
print("\nDissolved by node type:")
print(dissolved[["throughput", "cost"]].to_string())


# 5. High-throughput nodes
high_tp = gdf.where([value > 1100 for value in gdf["throughput"]])
print(f"\nHigh-throughput nodes (>1100): {list(high_tp['node'])}")


# 6. Bounding box: central corridor
central = gdf.cx[-111.90:-111.78, 40.73:40.77]
print(f"Nodes in central corridor: {list(central['node'])}")


# 7. Overlay: buffer intersections
buf_union = gdf.select_columns(["node"]).to_crs("EPSG:3857").buffer(5000).to_crs("EPSG:4326")
buf_rows = buf_union.to_records()
left_buf = gp.GeoPromptFrame.from_records(buf_rows[:3], crs=buf_union.crs)
right_buf = gp.GeoPromptFrame.from_records(buf_rows[1:4], crs=buf_union.crs)
overlaps = left_buf.overlay_intersections(right_buf, rsuffix="ov")
print(f"\nBuffer overlap areas: {len(overlaps)}")


gdf.to_file(str(OUTPUT_DIR / "d4-gp-network.geojson"), driver="GeoJSON")
print("\nWrote d4-gp-network.geojson")


# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

node_rows = gdf.to_records()
node_coords = [row["geometry"]["coordinates"] for row in node_rows]
node_x = [coord[0] for coord in node_coords]
node_y = [coord[1] for coord in node_coords]
scatter = axes[0].scatter(node_x, node_y, c=gdf["cost"], cmap="RdYlGn_r", s=160)
fig.colorbar(scatter, ax=axes[0], label="Routing Cost")
for row in node_rows:
    x, y = row["geometry"]["coordinates"]
    axes[0].annotate(row["node"], (x, y), textcoords="offset points", xytext=(4, 4), fontsize=9, fontweight="bold")

incident_rows = incidents_gdf.to_records()
incident_coords = [row["geometry"]["coordinates"] for row in incident_rows]
axes[0].scatter(
    [coord[0] for coord in incident_coords],
    [coord[1] for coord in incident_coords],
    color="red",
    s=100,
    marker="x",
    label="Incidents",
)
axes[0].set_title("Transport Network (x=incidents)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

gdf_s = gdf.sort_values("cost")
axes[1].barh(gdf_s["node"], gdf_s["cost"], color="#0891b2")
axes[1].set_xlabel("Routing Cost")
axes[1].set_title("Node Reachability Costs")
axes[1].grid(True, axis="x", alpha=0.3)

plt.suptitle("D4 Transportation: GeoPrompt Spatial Analysis", fontweight="bold")
plt.tight_layout()
plt.show()

# Basemap snapshot (real tiled basemap, saved as HTML)
try:
    import folium

    label_candidates = ["node", "stand_id", "asset_id", "zone_id"]
    label_col = next((c for c in label_candidates if c in gdf.columns), gdf.columns[0])
    centroids = gdf.geom.centroid()
    center_lat = float(np.mean([pt[1] for pt in centroids]))
    center_lon = float(np.mean([pt[0] for pt in centroids]))
    fmap = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles="CartoDB positron")

    for row in gdf.to_records():
        lon, lat = row["geometry"]["coordinates"]
        folium.CircleMarker(
            location=[float(lat), float(lon)],
            radius=6,
            color="#1d4ed8",
            fill=True,
            fill_opacity=0.85,
            popup=f"{label_col}: {row[label_col]}",
        ).add_to(fmap)

    if "incidents_gdf" in locals():
        for row in incidents_gdf.to_records():
            lon, lat = row["geometry"]["coordinates"]
            folium.Marker(
                location=[float(lat), float(lon)],
                icon=folium.Icon(color="red", icon="warning-sign", prefix="glyphicon"),
                popup=f"inc_id: {row.get('inc_id', 'n/a')}",
            ).add_to(fmap)

    optional_marker_specs = [
        ("demand_gdf", "demand_id", "blue", "bolt", "fa"),
        ("stations_gdf", "station_id", "green", "tree", "fa"),
        ("incidents_gdf", "inc_id", "red", "warning-sign", "glyphicon"),
        ("adapt_gdf", "site_id", "purple", "leaf", "fa"),
    ]

    for frame_name, id_key, color, icon, prefix in optional_marker_specs:
        frame = globals().get(frame_name)
        if frame is None:
            continue
        for row in frame.to_records():
            lon, lat = row["geometry"]["coordinates"]
            folium.Marker(
                location=[float(lat), float(lon)],
                icon=folium.Icon(color=color, icon=icon, prefix=prefix),
                popup=f"{id_key}: {row.get(id_key, 'n/a')}"
            ).add_to(fmap)

    map_path = OUTPUT_DIR / "d4-gp-basemap.html"
    fmap.save(str(map_path))
    print(f"Wrote basemap snapshot: {map_path.name}")
    from IPython.display import display
    display(fmap)
except Exception as exc:
    print(f"Basemap snapshot skipped: {exc}")

Transport nodes:
node  cost  throughput
   O   0.0        1500
   A   5.0        1200
   B   9.0        1100
   C  12.0         900
   D  14.0         800
   E  11.0         950
   F   3.0        1300

Service buffers (3km): 7

Incidents assigned to nearest node (km):
inc_id  severity node  dist_km
    I1         3    A 1.843577
    I2         1    D 1.843049
    I3         2    C 1.113195
Incidents within 3km service zones:
inc_id node
    I1    O
    I1    A
    I2    D
    I2    E
    I3    C

Dissolved by node type:
           throughput  cost
node_type                  
dest              900  12.0
inter            5350   8.4
origin           1500   0.0

High-throughput nodes (>1100): ['O', 'A', 'F']
Nodes in central corridor: ['O', 'A', 'B', 'C']

Buffer overlap areas: 7

Wrote d4-gp-network.geojson
Wrote basemap snapshot: d4-gp-basemap.html


C:\Users\Matt\AppData\Local\Temp\ipykernel_2668\1217816665.py:118: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section C: Scenario Comparison


In [4]:
scenarios = {

    "no_action":    {"throughput": 1000, "avg_delay_min": 18.0, "cost_musd": 0.0},

    "signal_opt":   {"throughput": 1280, "avg_delay_min": 11.0, "cost_musd": 15.0},

    "new_corridor": {"throughput": 1600, "avg_delay_min":  7.0, "cost_musd": 85.0},

}

scenario_records = []

for name, vals in scenarios.items():

    score = round(vals["throughput"] / 1600 * 0.5 + (1.0 / vals["avg_delay_min"]) * 10 * 0.3

                  + (1.0 / max(vals["cost_musd"] + 1, 1)) * 20 * 0.2, 4)

    scenario_records.append({"scenario": name, **vals, "score": score})

scenario_records.sort(key=lambda r: -float(r["score"]))



fig, axes = plt.subplots(1, 3, figsize=(14, 4))

names = [r["scenario"] for r in scenario_records]

colors = ["#27ae60", "#e67e22", "#c0392b"]

axes[0].barh(names, [r["throughput"] for r in scenario_records], color=colors)

axes[0].set_xlabel("Throughput"); axes[0].set_title("Network Throughput"); axes[0].grid(True, axis="x", alpha=0.3)

axes[1].barh(names, [r["avg_delay_min"] for r in scenario_records], color=colors)

axes[1].set_xlabel("Avg Delay (min)"); axes[1].set_title("Average Delay"); axes[1].grid(True, axis="x", alpha=0.3)

axes[2].barh(names, [r["score"] for r in scenario_records], color=colors)

axes[2].set_xlabel("Composite Score"); axes[2].set_title("Scenario Score"); axes[2].grid(True, axis="x", alpha=0.3)

plt.suptitle("D4 Transportation (GeoPrompt): Scenario Comparison", fontweight="bold")

plt.tight_layout(); plt.show()



(OUTPUT_DIR / "d4-gp-complex.json").write_text(

    json.dumps({"scenario_ranking": scenario_records}, indent=2, default=str), encoding="utf-8"

)

print("Wrote d4-gp-complex.json")

for r in scenario_records:

    print(f"  {r['scenario']}: score={r['score']}")


Wrote d4-gp-complex.json
  no_action: score=4.4792
  new_corridor: score=0.9751
  signal_opt: score=0.9227


C:\Users\Matt\AppData\Local\Temp\ipykernel_5164\4109759037.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
